# Data Demo: Predicting 3-day VIX change from Fed text

This notebook demonstrates the project's data loader on a small bundled subset of the corpus (~10 representative documents). It runs end-to-end with **no network access, no FRED API key, and no Talapas access required**: the example data lives at `data/example/` in this repo.

**Inputs:** the first 80 sentences of a Federal Reserve document, encoded one-at-a-time with a frozen FinBERT (`yiyanghkust/finbert-pretrain`) and mask-weighted mean-pooled into one 768-d vector per sentence.

**Target:** the 3-day close-to-close change in the CBOE VIX (FRED ticker `VIXCLS`) following the document's release date, aligned to the next available trading day when a release falls on a weekend or U.S. market holiday.

**Sources:** FOMC minutes (1993 to present, ~268 docs) and Humphrey-Hawkins / Semiannual Monetary Policy Report testimony (1997 to present, ~58 docs). 326 docs total in the full corpus; this notebook shows a 10-doc demo subset.

## 0. Install the project package

If you have already run `pip install -e .` in this kernel's Python, this cell is a no-op. Otherwise it installs the package in editable mode so the imports below work. Safe to re-run.

In [3]:
import sys, subprocess
from pathlib import Path

# Find repo root by walking up until we see pyproject.toml.
_here = Path.cwd()
for candidate in [_here, *list(_here.parents)]:
    if (candidate / "pyproject.toml").exists():
        repo_root = candidate
        break
else:
    raise RuntimeError("Could not find repo root: no pyproject.toml in CWD or any parent directory.")

try:
    import transcripts_fed_vix  # noqa: F401
    print(f"Package already installed in this kernel.")
except ModuleNotFoundError:
    print(f"Installing transcripts_fed_vix from {repo_root} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root)])
    print("Done. Note: you may need to restart the kernel for fresh imports to take effect.")

Package already installed in this kernel.


## 1. Load the example documents

`load_example_documents()` reads the bundled JSON and returns a DataFrame whose schema matches the full processed parquet: `doc_id`, `source`, `release_date`, `aligned_trading_date`, `vix_t`, `vix_t_plus_3`, `target`, `sentences`.

In [4]:
import pandas as pd
import numpy as np
import torch

from transcripts_fed_vix.data import load_example_documents, get_example_dataloader

documents = load_example_documents()
print(f"Loaded {len(documents)} example documents.")
print(f"Date range: {documents['release_date'].min().date()} .. {documents['release_date'].max().date()}")
documents[["doc_id", "source", "release_date", "vix_t", "vix_t_plus_3", "target"]]

FileNotFoundError: Example data missing: /Users/zoetomlinson/Desktop/Personal Projects/Transcripts-Fed-VIX-DL/notebooks/data/example/example_documents.json. Run scripts/export_examples.py on the cluster (or wherever the full processed parquet lives) to generate the bundled examples.

## 2. Inspect one document's text

Each row's `sentences` column is a list of strings (first 20 here in the bundled subset; up to 80 in the full corpus; see methodology doc for why).

In [ ]:
row = documents.iloc[0]
print(f"Document: {row['doc_id']}")
print(f"  source:               {row['source']}")
print(f"  release_date:         {row['release_date'].date()}")
print(f"  aligned_trading_date: {row['aligned_trading_date'].date()}")
print(f"  vix_t:                {row['vix_t']:.2f}")
print(f"  vix_t_plus_3:         {row['vix_t_plus_3']:.2f}")
print(f"  target (3-day change): {row['target']:+.2f}")
print(f"  number of sentences:  {len(row['sentences'])}")
print("\nFirst 3 sentences:")
for i, s in enumerate(row["sentences"][:3], 1):
    print(f"  [{i}] {s}")

## 3. The training DataLoader: one batch end-to-end

`get_example_dataloader()` returns a standard `torch.utils.data.DataLoader`. Each batch is a dict containing the pre-computed FinBERT embeddings, an attention mask, the regression target, and metadata (doc IDs and release dates).

Embeddings are pre-computed because the FinBERT encoder is fully frozen, so there's no information-leakage risk to caching them once and reusing them every training epoch. This speeds training by orders of magnitude (we only train the small attention aggregator + linear head, ~100k parameters, on top of the cached embeddings).

In [ ]:
loader = get_example_dataloader(batch_size=4)
batch = next(iter(loader))

print("Batch keys:", list(batch.keys()))
print(f"  embeddings:    {tuple(batch['embeddings'].shape)}  dtype={batch['embeddings'].dtype}")
print(f"  mask:          {tuple(batch['mask'].shape)}        dtype={batch['mask'].dtype}")
print(f"  target:        {tuple(batch['target'].shape)}        dtype={batch['target'].dtype}")
print(f"  doc_ids:       {batch['doc_ids']}")
print(f"  release_dates: {[d.date() for d in batch['release_dates']]}")
print(f"  target values: {batch['target'].tolist()}")

## 4. Target distribution in the demo subset

Each document's target is the *change* in the VIX over the 3 trading days following its release. Positive = volatility went up after the Fed spoke; negative = it went down.

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(documents["doc_id"], documents["target"], color="#4C78A8")
ax.axhline(0, color="gray", linewidth=0.8, alpha=0.5)
ax.set_ylabel("3-day VIX change (target)")
ax.set_xlabel("Document")
ax.set_title("Targets for the 10 demo documents")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

## 5. Optional: run the trained model on the demo docs and visualize attention

If `data/example/example_model.pt` is present (the trained checkpoint exported alongside the demo data), we can run a forward pass and see which sentences the attention layer weighted most heavily for one document. This is purely illustrative; full results, residual analyses, and Chow tests live in `outputs/` after running the full pipeline on Talapas.

In [ ]:
from transcripts_fed_vix.models import SentenceAttentionModel
from transcripts_fed_vix.models.attention import AttentionConfig

model_path = repo_root / "data" / "example" / "example_model.pt"
if model_path.exists():
    model = SentenceAttentionModel(AttentionConfig())
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    with torch.no_grad():
        out = model(batch["embeddings"], batch["mask"])
    for i, doc_id in enumerate(batch["doc_ids"]):
        n_real = int(batch["mask"][i].sum())
        print(f"\n{doc_id}  target={batch['target'][i].item():+.2f}  pred={out.prediction[i].item():+.2f}")
        top = torch.topk(out.attention_weights[i, :n_real], k=min(3, n_real))
        print("  Top-3 attended sentences:")
        sents = documents.set_index("doc_id").loc[doc_id, "sentences"]
        for idx, w in zip(top.indices.tolist(), top.values.tolist()):
            print(f"    weight={w:.3f}  {sents[idx][:120]}")
else:
    print(f"No trained model at {model_path}; skipping this cell. "
          "Run scripts/export_examples.py on Talapas after training to bundle one.")

## Next steps

- Full scrape + training pipeline lives in `scripts/train.sbatch` (SLURM job)
- Methodology and design-decision rationale: `docs/METHODOLOGY.md`
- Project milestone summary: `docs/MILESTONE_REPORT.md`